# Issue 7: Análisis de componentes principales (PCA)

Continúa [issue6.ipynb](issue6.ipynb), que ejecuta los issues anteriores de la cadena y deja `X_train`, `y_train`, `X_val`, `y_val`, `X_test` y `y_test` con las numéricas winsorizadas, con `log1p` en las asimétricas, imputadas con la mediana (provisional) y escaladas con `StandardScaler`.

**Objetivo:** decidir si conviene reemplazar las variables numéricas por componentes principales antes del modelado. PCA no mira el target: busca las direcciones de mayor varianza, que no tienen por qué ser las que mejor separan `Charged Off`.

**Reglas:** PCA se ajusta solo con entrenamiento; las comparaciones se hacen en validación; el conjunto de prueba no se usa. `issue_d` no entra, porque la propuesta no lo admite como predictor.

In [1]:
%run issue6.ipynb

Filas en df_work: 1,348,059
Columnas en df_work: 104
¿Existe 'target' en df_work?: True
Columnas excluidas por src/data.py: 48
Columnas excluidas que siguen presentes: []
Filas totales: 1,348,059
Columnas totales: 104

=== TIPOS DE DATOS ===
float64           86
str               15
datetime64[us]     1
int8               1
category           1
Name: count, dtype: int64


target
0    1078739
1     269320
Name: count, dtype: int64
Tasa de Charged Off: 0.1998


Entrenamiento: 862,757 filas, 101 columnas, Charged Off 19.98%
Validación: 215,690 filas, 101 columnas, Charged Off 19.98%
Prueba: 269,612 filas, 101 columnas, Charged Off 19.98%


Columnas finales: 76
X_train: (862757, 76) | X_val: (215690, 76) | X_test: (269612, 76)
Entrenamiento: 862,757 préstamos, 100 columnas candidatas
Charged Off en entrenamiento: 19.98%
numérica            85
categórica/texto    15
Ninguna de las 48 columnas excluidas está en X_ent.


Columnas 100 % vacías: ninguna

Casi constantes (dominancia >= 99%); tasa global de Charged Off = 19.98%
36 columnas con 50 % a 99.9 % de nulos (quedan para el Issue 5):
Figura guardada en reports/figures/redundancia_dominancia.png


/var/folders/5c/7vn7bdh16xj7r0pwx0xs9zjm0000gn/T/ipykernel_45368/138448747.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


... (24 outputs left out)

Pasos: ['winsorizacion', 'log1p', 'imputacion_provisional', 'escalado']
Columnas con log1p: 27; con winsorización: 61
NaN tras el pipeline -> entrenamiento: 0, validación: 0


Entrenamiento: medias en [-9.7e-16, 1.7e-15], desviaciones en [1.0000, 1.0000]
Validación (sin reajustar): medias en [-0.006, 0.004], desviaciones en [0.990, 1.003]


Sin imputador: los NaN de validación se conservan en las mismas posiciones.


Numéricas tratadas: 61; categóricas sin cambios: 8
X_train: (862757, 69) | X_val: (215690, 69) | X_test: (269612, 69)
NaN en numéricas -> X_train: 0, X_val: 0, X_test: 0


In [2]:
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import TwoSlopeNorm
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

sys.path.append(str(Path.cwd().parent))
from src.data import COLUMNA_FECHA, SEED

pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 120)

DIR_FIGURAS = Path.cwd().parent / 'reports' / 'figures'
AZUL, NARANJA, TINTA, TINTA_2, REJILLA = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'axes.edgecolor': TINTA_2,
    'axes.labelcolor': TINTA, 'xtick.color': TINTA_2, 'ytick.color': TINTA_2,
    'axes.grid': True, 'grid.color': REJILLA, 'grid.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10,
    'axes.titlesize': 11, 'axes.titleweight': 'bold',
})

def guardar(fig, nombre):
    fig.savefig(DIR_FIGURAS / nombre, dpi=150, bbox_inches='tight')
    print(f'Figura guardada en reports/figures/{nombre}')

## 1. Datos de la cadena

PCA se aplica a las columnas numéricas que deja `issue6.ipynb`. Las categóricas (`term`, `purpose`, `addr_state`, etc.) aún no están codificadas y no entran: PCA sobre variables one-hot mezclaría escalas sin sentido y la codificación no es parte de este issue.

Antes de ajustar se comprueba lo que PCA necesita: sin `issue_d`, sin nulos y con varianza 1 en todas las columnas. Si una columna tuviera más varianza que el resto, dominaría los primeros componentes solo por su escala.

In [3]:
numericas = list(X_train.select_dtypes('number').columns)
categoricas = [c for c in X_train.columns if c not in numericas]
X_ent = X_train[numericas]      # nombres de trabajo, sin pisar los de la cadena
X_val_num = X_val[numericas]

assert COLUMNA_FECHA not in X_train.columns, 'issue_d no puede entrar al PCA'
assert X_ent.isna().sum().sum() == 0 and X_val_num.isna().sum().sum() == 0
desv = X_ent.std()
assert np.allclose(desv, 1, atol=1e-3) and np.allclose(X_ent.mean(), 0, atol=1e-6)

print(f'Numéricas para PCA: {len(numericas)}; categóricas fuera del PCA: {len(categoricas)} {categoricas}')
print(f'Entrenamiento: {X_ent.shape}; validación: {X_val_num.shape}')
print(f'Desviaciones en entrenamiento: [{desv.min():.4f}, {desv.max():.4f}]')

Numéricas para PCA: 61; categóricas fuera del PCA: 8 ['term', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'earliest_cr_line', 'application_type']
Entrenamiento: (862757, 61); validación: (215690, 61)
Desviaciones en entrenamiento: [1.0000, 1.0000]


## 2. Varianza explicada

Se ajusta PCA con todos los componentes, solo con entrenamiento, para ver cómo se reparte la varianza. Se usan dos criterios habituales para elegir el número de componentes:

- **Varianza acumulada:** el menor número de componentes que alcanza 80 %, 90 %, 95 % o 99 %.
- **Criterio de Kaiser:** conservar los componentes con autovalor mayor que 1, es decir, los que explican más varianza que una variable original estandarizada.

In [4]:
pca = PCA(random_state=SEED).fit(X_ent)
nombres_pc = [f'PC{i + 1}' for i in range(pca.n_components_)]
varianza = pd.DataFrame({
    'autovalor': pca.explained_variance_,
    'varianza_%': pca.explained_variance_ratio_ * 100,
    'acumulada_%': pca.explained_variance_ratio_.cumsum() * 100,
}, index=nombres_pc)

UMBRALES = (80, 90, 95, 99)
componentes_para = {u: int(np.searchsorted(varianza['acumulada_%'].values, u) + 1) for u in UMBRALES}
kaiser = int((varianza['autovalor'] > 1).sum())

print('Componentes necesarios por varianza acumulada:')
for u, k in componentes_para.items():
    print(f'  {u} % -> {k} de {len(numericas)}')
print(f'Criterio de Kaiser (autovalor > 1): {kaiser} componentes, {varianza["acumulada_%"].iloc[kaiser - 1]:.1f} % de la varianza')
varianza.head(12).round(3)

Componentes necesarios por varianza acumulada:
  80 % -> 22 de 61
  90 % -> 32 de 61
  95 % -> 41 de 61
  99 % -> 53 de 61
Criterio de Kaiser (autovalor > 1): 17 componentes, 73.5 % de la varianza


,autovalor,varianza_%,acumulada_%
PC1,9.005,14.762,14.762
PC2,5.434,8.908,23.670
PC3,4.897,8.027,31.697
PC4,3.765,6.171,37.869
PC5,3.353,5.497,43.366
PC6,2.554,4.187,47.553
PC7,2.213,3.627,51.180
PC8,1.933,3.169,54.349
PC9,1.858,3.045,57.394
PC10,1.505,2.468,59.862


In [5]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3.8))
x = np.arange(1, pca.n_components_ + 1)

ax1.bar(x, varianza['autovalor'], color=AZUL, width=0.8)
ax1.axhline(1, color=NARANJA, linestyle='--', linewidth=1)
ax1.text(pca.n_components_, 1, 'Kaiser: autovalor = 1', ha='right', va='bottom', fontsize=8, color=NARANJA)
ax1.set_xlabel('Componente')
ax1.set_ylabel('Autovalor')
ax1.set_title('Gráfico de sedimentación')
ax1.grid(axis='x', visible=False)

ax2.plot(x, varianza['acumulada_%'], color=AZUL, linewidth=2)
for u, k in componentes_para.items():
    ax2.plot([k, k], [0, u], color=REJILLA, linewidth=1, zorder=0)
    ax2.scatter([k], [u], color=TINTA, s=18, zorder=3)
    ax2.annotate(f'{u} %: {k}', (k, u), textcoords='offset points', xytext=(6, -12), fontsize=8, color=TINTA_2)
ax2.set_ylim(0, 102)
ax2.set_xlabel('Número de componentes')
ax2.set_ylabel('Varianza acumulada (%)')
ax2.set_title('Varianza explicada acumulada')

fig.suptitle(f'PCA sobre las {len(numericas)} numéricas (ajustado con entrenamiento)', fontweight='bold')
fig.tight_layout()
guardar(fig, 'pca_varianza_explicada.png')
plt.show()

Figura guardada en reports/figures/pca_varianza_explicada.png


/var/folders/5c/7vn7bdh16xj7r0pwx0xs9zjm0000gn/T/ipykernel_45368/3775949537.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura:** la varianza está repartida. El primer componente explica solo un 15 % y no hay un codo claro en el gráfico de sedimentación. Hacen falta 22 componentes para el 80 %, 32 para el 90 % y 41 para el 95 %. Kaiser se queda con 17.

Esto se explica por cómo quedaron los datos tras los issues 3 y 6: ya se eliminaron los pares casi idénticos (correlación ≥ 0.9), así que las 61 columnas restantes aportan bastante información propia. PCA no tiene mucha redundancia lineal que comprimir.

## 3. Cargas de los primeros componentes

Las cargas son los coeficientes de cada variable original en un componente. Se muestran las variables con mayor carga absoluta en los cinco primeros componentes para darles una interpretación.

In [6]:
cargas = pd.DataFrame(pca.components_.T, index=numericas, columns=nombres_pc)

def top_cargas(componente, n=6):
    s = cargas[componente]
    orden = s.abs().sort_values(ascending=False).index[:n]
    return ', '.join(f'{col} ({s[col]:+.2f})' for col in orden)

pd.DataFrame({
    'varianza_%': varianza.loc[nombres_pc[:5], 'varianza_%'].round(1),
    'variables con mayor carga': [top_cargas(pc) for pc in nombres_pc[:5]],
})

,varianza_%,variables con mayor carga
PC1,14.8,"open_acc (+0.26), num_op_rev_tl (+0.26), num_rev_accts (+0.24), total_acc (+0.24), total_rev_hi_lim (+0.24), num_bc_..."
PC2,8.9,"mo_sin_rcnt_rev_tl_op (+0.24), revol_bal (+0.24), num_tl_op_past_12m (-0.24), mo_sin_rcnt_tl (+0.22), open_rv_12m (-..."
PC3,8.0,"all_util (+0.26), num_il_tl (+0.25), open_act_il (+0.25), open_il_24m (+0.25), open_il_12m (+0.23), mths_since_rcnt_..."
PC4,6.2,"bc_util (+0.36), revol_util (+0.34), percent_bc_gt_75 (+0.33), fico_range_low (-0.32), num_actv_rev_tl (+0.28), bc_o..."
PC5,5.5,"delinq_2yrs (+0.43), mths_since_last_delinq (-0.38), num_tl_90g_dpd_24m (+0.35), mths_since_last_major_derog (-0.35)..."


In [7]:
PRIMEROS = nombres_pc[:5]
seleccion = sorted({col for pc in PRIMEROS for col in cargas[pc].abs().nlargest(6).index},
                   key=lambda col: (cargas.loc[col, PRIMEROS].abs().values.argmax(), -cargas.loc[col, PRIMEROS].abs().max()))
datos = cargas.loc[seleccion, PRIMEROS]

fig, ax = plt.subplots(figsize=(7, 0.32 * len(seleccion) + 1.2))
lim = np.abs(datos.values).max()
im = ax.imshow(datos.values, cmap='RdBu_r', norm=TwoSlopeNorm(0, -lim, lim), aspect='auto')
ax.set_xticks(range(len(PRIMEROS)), [f'{pc}\n{varianza.loc[pc, "varianza_%"]:.1f} %' for pc in PRIMEROS])
ax.set_yticks(range(len(seleccion)), seleccion)
for i in range(len(seleccion)):
    for j in range(len(PRIMEROS)):
        v = datos.values[i, j]
        ax.text(j, i, f'{v:+.2f}', ha='center', va='center', fontsize=7,
                color='white' if abs(v) > 0.6 * lim else TINTA)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.6, label='carga')
ax.set_title('Cargas de las variables principales en PC1–PC5')
fig.tight_layout()
guardar(fig, 'pca_cargas.png')
plt.show()

Figura guardada en reports/figures/pca_cargas.png


/var/folders/5c/7vn7bdh16xj7r0pwx0xs9zjm0000gn/T/ipykernel_45368/2621225767.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretación de los cinco primeros componentes:**

| Componente | Variables con mayor carga | Lectura |
|---|---|---|
| PC1 (≈15 %) | `open_acc`, `num_op_rev_tl`, `num_rev_accts`, `total_acc`, `total_rev_hi_lim`, `num_bc_sats` (todas +) | **Tamaño del historial crediticio:** cuántas cuentas tiene el solicitante y cuánto crédito rotativo le dieron. |
| PC2 (≈9 %) | `mo_sin_rcnt_rev_tl_op`, `revol_bal`, `mo_sin_rcnt_tl` (+) frente a `num_tl_op_past_12m`, `open_rv_12m`, `open_rv_24m` (−) | **Actividad reciente:** opone a quien abrió cuentas hace poco con quien lleva tiempo sin abrir ninguna. |
| PC3 (≈8 %) | `all_util`, `num_il_tl`, `open_act_il`, `open_il_24m`, `open_il_12m` (+), `mths_since_rcnt_il` (−) | **Préstamos a plazos (installment):** cantidad y uso de créditos no rotativos. |
| PC4 (≈6 %) | `bc_util`, `revol_util`, `percent_bc_gt_75` (+) frente a `fico_range_low`, `bc_open_to_buy` (−) | **Uso del crédito rotativo frente al puntaje FICO:** tarjetas cerca del límite y FICO bajo. Es el perfil de riesgo clásico. |
| PC5 (≈5 %) | `delinq_2yrs`, `num_tl_90g_dpd_24m` (+) frente a los `mths_since_*delinq*` y `mths_since_last_major_derog` (−) | **Morosidad pasada:** atrasos recientes y pocos meses desde el último. |

Los componentes tienen sentido crediticio, pero el de más varianza (PC1, tamaño del historial) no es el más relacionado con el riesgo. La siguiente sección lo mide.

## 4. ¿Los componentes con más varianza son los que más predicen?

Para cada componente se calcula, en entrenamiento, su ROC-AUC univariado frente a `Charged Off`. Se toma `max(AUC, 1 − AUC)` porque el signo de un componente es arbitrario. Un valor de 0.5 significa que el componente, por sí solo, no separa las clases.

In [8]:
Z_ent = pd.DataFrame(pca.transform(X_ent), columns=nombres_pc, index=X_ent.index)
Z_val = pd.DataFrame(pca.transform(X_val_num), columns=nombres_pc, index=X_val_num.index)

auc_ent = Z_ent.apply(lambda z: roc_auc_score(y_train, z))
varianza['auc_univariado'] = np.maximum(auc_ent, 1 - auc_ent)
varianza['rango_auc'] = varianza['auc_univariado'].rank(ascending=False).astype(int)

print('Componentes con mayor AUC univariado en entrenamiento:')
varianza.sort_values('auc_univariado', ascending=False).head(10)[['varianza_%', 'auc_univariado']].round(3)

Componentes con mayor AUC univariado en entrenamiento:


,varianza_%,auc_univariado
PC4,6.171,0.587
PC25,1.068,0.562
PC2,8.908,0.559
PC28,0.938,0.547
PC3,8.027,0.541
PC5,5.497,0.538
PC23,1.190,0.536
PC8,3.169,0.534
PC33,0.727,0.533
PC30,0.854,0.532


In [9]:
fig, ax = plt.subplots(figsize=(14, 3.8))
ax.bar(x, varianza['auc_univariado'] - 0.5, bottom=0.5, color=AZUL, width=0.8, label='AUC univariado')
ax.set_ylim(0.5, varianza['auc_univariado'].max() + 0.01)
ax.set_xlabel('Componente (ordenado por varianza explicada, de mayor a menor)')
ax.set_ylabel('ROC-AUC univariado')
ax.grid(axis='x', visible=False)
for pc in varianza['auc_univariado'].nlargest(3).index:
    k = nombres_pc.index(pc) + 1
    ax.annotate(pc, (k, varianza.loc[pc, 'auc_univariado']), textcoords='offset points', xytext=(0, 3),
                ha='center', fontsize=8, color=TINTA)
ax2 = ax.twinx()
ax2.plot(x, varianza['varianza_%'], color=NARANJA, linewidth=1.5, label='varianza explicada')
ax2.set_ylabel('Varianza explicada (%)', color=NARANJA)
ax2.tick_params(axis='y', colors=NARANJA)
ax2.grid(False)
ax2.spines['right'].set_visible(True)
ax.set_title('Varianza explicada frente a capacidad predictiva de cada componente (entrenamiento)')
fig.tight_layout()
guardar(fig, 'pca_varianza_vs_auc.png')
plt.show()

Figura guardada en reports/figures/pca_varianza_vs_auc.png


/var/folders/5c/7vn7bdh16xj7r0pwx0xs9zjm0000gn/T/ipykernel_45368/3587468170.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura:** la capacidad predictiva no sigue a la varianza. PC4 (uso del crédito frente a FICO) es el componente más predictivo, aunque solo explica un 6 % de la varianza. PC1, el de más varianza, tiene un AUC prácticamente de 0.50: no separa las clases. Además, aparecen componentes "menores" entre los más predictivos, como PC25 (segundo en AUC), que explica apenas un 1 % de la varianza y está dominado por `open_act_il`, `loan_amnt` y `dti`.

Esto anticipa el problema de PCA en este proyecto: si se corta por varianza acumulada, se pueden descartar justamente direcciones con señal sobre el target.

## 5. Comparación en validación

Se entrena la misma regresión logística del baseline (`max_iter=1000`, `random_state=SEED`) con:

- **Referencia:** las numéricas originales, sin PCA.
- **PCA, primeros k componentes:** el uso habitual, cortando por varianza. Se prueban varios valores de k, incluidos los de los umbrales de varianza y de Kaiser.
- **PCA, k componentes elegidos por AUC univariado en entrenamiento:** una variante que ordena los componentes por su relación con el target en lugar de por varianza. Sirve para ver cuánto se pierde por cortar solo por varianza.

Las categóricas no entran en ninguno de los casos, así que la comparación es justa entre sí; los valores no son comparables con un modelo que sí las use.

In [10]:
def evaluar(X_ajuste, X_eval):
    modelo = LogisticRegression(max_iter=1000, random_state=SEED)
    inicio = time.time()
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter('always', ConvergenceWarning)
        modelo.fit(X_ajuste, y_train)
    prob = modelo.predict_proba(X_eval)[:, 1]
    return {
        'ROC-AUC': roc_auc_score(y_val, prob),
        'PR-AUC': average_precision_score(y_val, prob),
        'iteraciones': int(modelo.n_iter_[0]),
        'convergió': not any(issubclass(a.category, ConvergenceWarning) for a in avisos),
        'segundos': round(time.time() - inicio, 1),
    }

referencia = evaluar(X_ent, X_val_num)

VALORES_K = sorted({5, 10, 15, kaiser, 20, 25, 30, 35, *componentes_para.values(), 45, 50, 55, len(numericas)})
por_auc = list(varianza.sort_values('auc_univariado', ascending=False).index)

filas = [{'método': 'Sin PCA (referencia)', 'k': len(numericas), 'varianza_%': 100.0, **referencia}]
for k in VALORES_K:
    filas.append({'método': 'PCA: primeros k', 'k': k, 'varianza_%': varianza['acumulada_%'].iloc[k - 1],
                  **evaluar(Z_ent.iloc[:, :k], Z_val.iloc[:, :k])})
    columnas = por_auc[:k]
    filas.append({'método': 'PCA: k por AUC', 'k': k, 'varianza_%': varianza.loc[columnas, 'varianza_%'].sum(),
                  **evaluar(Z_ent[columnas], Z_val[columnas])})
comparacion = pd.DataFrame(filas)
comparacion['Δ ROC-AUC'] = comparacion['ROC-AUC'] - referencia['ROC-AUC']
comparacion['Δ PR-AUC'] = comparacion['PR-AUC'] - referencia['PR-AUC']
comparacion.set_index(['método', 'k']).round(4)

,,varianza_%,ROC-AUC,PR-AUC,iteraciones,convergió,segundos,Δ ROC-AUC,Δ PR-AUC
método,k,,,,,,,,
Sin PCA (referencia),61,100.0000,0.6843,0.3419,36,True,2.4,0.0000,0.0000
PCA: primeros k,5,43.3660,0.6195,0.2784,7,True,0.5,-0.0647,-0.0634
PCA: k por AUC,5,25.1129,0.6396,0.2983,11,True,0.7,-0.0447,-0.0436
PCA: primeros k,10,59.8621,0.6282,0.2878,7,True,0.4,-0.0561,-0.0540
PCA: k por AUC,10,36.5499,0.6578,0.3158,13,True,0.6,-0.0265,-0.0261
PCA: primeros k,15,70.0687,0.6380,0.2932,10,True,0.6,-0.0463,-0.0486
PCA: k por AUC,15,44.2540,0.6676,0.3250,13,True,0.7,-0.0166,-0.0169
PCA: primeros k,17,73.5328,0.6384,0.2940,10,True,0.5,-0.0459,-0.0479
PCA: k por AUC,17,46.8514,0.6715,0.3271,14,True,0.8,-0.0128,-0.0148


In [11]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 3.8), sharex=True)
for ax, metrica in zip(ejes, ['ROC-AUC', 'PR-AUC']):
    for metodo, color in (('PCA: primeros k', AZUL), ('PCA: k por AUC', NARANJA)):
        datos = comparacion[comparacion['método'] == metodo]
        ax.plot(datos['k'], datos[metrica], color=color, marker='o', markersize=4, linewidth=1.5, label=metodo)
    ax.axhline(referencia[metrica], color=TINTA_2, linestyle='--', linewidth=1)
    ax.text(5, referencia[metrica], f'sin PCA {referencia[metrica]:.4f}', ha='left', va='bottom', fontsize=8, color=TINTA_2)
    for u, k in componentes_para.items():
        if u in (90, 95):
            ax.axvline(k, color=REJILLA, linewidth=1, zorder=0)
            ax.text(k, ax.get_ylim()[0], f' {u} %', fontsize=8, color=TINTA_2, va='bottom')
    ax.set_xlabel('Número de componentes (k)')
    ax.set_ylabel(f'{metrica} en validación')
    ax.set_title(metrica)
ejes[1].legend(loc='lower right', frameon=False)
fig.suptitle('Regresión logística en validación: numéricas originales frente a componentes principales', fontweight='bold')
fig.tight_layout()
guardar(fig, 'pca_comparacion_validacion.png')
plt.show()

Figura guardada en reports/figures/pca_comparacion_validacion.png


/var/folders/5c/7vn7bdh16xj7r0pwx0xs9zjm0000gn/T/ipykernel_45368/2654492599.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura:**

- **Ningún k mejora a las variables originales.** Con todos los componentes, el resultado es idéntico al de referencia: es solo una rotación de los mismos datos, y la regresión logística es invariante a rotaciones. Cortando por varianza, cualquier k menor pierde rendimiento.
- **Cortar por varianza cuesta caro con pocos componentes.** Con el criterio de Kaiser (17 componentes, 73.5 % de la varianza) el ROC-AUC cae de 0.6843 a 0.6384. Con 41 componentes (95 % de la varianza) la pérdida es pequeña, pero solo se ahorran 20 columnas.
- **El salto entre k = 20 y k = 25** coincide con la entrada de PC25, uno de los componentes más predictivos pese a su baja varianza (sección 4).
- **Ordenar por AUC recupera la señal antes** (con 20 componentes: 0.6758 frente a 0.6400), lo que confirma que el problema es el criterio de varianza y no la información. Aun así, solo iguala a la referencia desde k = 50, con diferencias de ±0.0002 que son ruido, y para entonces ya casi no reduce dimensiones. Además, esa variante ya no es PCA "puro": usa el target para elegir.

Una ventaja menor de PCA es que la regresión logística converge en menos iteraciones, porque los componentes son ortogonales. Con estos tamaños el modelo entrena en pocos segundos de todos modos, así que no compensa la pérdida.

## 6. Por qué no se usa LDA

El profesor recomendó también el análisis discriminante lineal (LDA). LDA, a diferencia de PCA, sí usa el target, pero produce como máximo *(número de clases − 1)* componentes. Con un target binario (`Charged Off` / pagado) eso es **un solo componente**: no sirve como técnica de reducción a varias dimensiones, sino que equivale a un clasificador lineal. Por eso queda fuera del preprocesamiento, como se acordó en el Issue 1.

## 7. Conclusiones

1. **PCA no se usa en el dataset final.** En validación, ningún número de componentes mejora a la regresión logística con las numéricas originales (ROC-AUC 0.6843, PR-AUC 0.3419). Con el 95 % de la varianza (41 componentes) se pierden 0.002 de ROC-AUC y con el criterio de Kaiser, 0.046.
2. **La redundancia lineal ya se eliminó en el Issue 3.** Las numéricas restantes reparten la varianza (PC1 ≈ 15 %, 41 componentes para el 95 %), así que PCA comprime poco.
3. **Varianza no es poder predictivo.** El componente más predictivo es PC4 (uso del crédito frente a FICO), y PC25, con apenas un 1 % de la varianza, es el segundo; PC1 no separa las clases. Cortar por varianza descarta señal.
4. **Se pierde interpretabilidad.** Las variables originales (`fico_range_low`, `revol_util`, `dti`) se explican directamente en un modelo de riesgo; los componentes, no.
5. **El análisis sigue siendo útil para el informe:** los cinco primeros componentes resumen la estructura de los datos (tamaño del historial, actividad reciente, préstamos a plazos, uso del crédito frente a FICO y morosidad pasada).

**Cuándo revisar esta decisión:** si se prueban modelos sensibles a la dimensión o a la colinealidad (por ejemplo, k-NN o SVM con kernel), puede valer la pena repetir la comparación con ellos. El objeto `pca` y la tabla `comparacion` quedan disponibles al ejecutar este notebook.

**Pendientes de la cadena que afectan a este resultado:**

- Cuando cierre el Issue 5, la imputación provisional con la mediana cambiará; conviene volver a ejecutar este notebook.
- La cadena entrega `loan_amnt` entre las numéricas, aunque `src/data.py` la marca como condicional (no se usa por defecto). Hay que confirmar con el equipo si entra.

## Salida

PCA no se aplica: `X_train`, `X_val` y `X_test` salen de este notebook **sin cambios** respecto de `issue6.ipynb`. Son el dataset final para los modelos: numéricas tratadas por el Issue 6 y categóricas aún sin codificar (pendiente para el modelado).

In [12]:
assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns)
assert X_train[numericas].equals(X_ent), 'Este notebook no debe modificar X_train'
print(f'X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}')

X_train: (862757, 69) | X_val: (215690, 69) | X_test: (269612, 69)
